In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run ../src/utils/reader_utils

In [0]:
%run ../src/utils/writer_utils

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today() - timedelta(days=1)

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
# Constants

SOURCE_TABLE_CONF = {
    "table": "shipment_events_raw",
    "schema": "bronze",
    "dedupe_asc": ["event_id"],
    "order_col": "event_timestamp",
    "timestamp_col": "_ingested_at"
}

TARGET_TABLE_CONF = {
    "table": "pyspark_shipment_events",
    "schema": "silver",
    "merge_keys": ["event_id"],
    "primary_key": "event_id"
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
shipment_events_df = read_delta_table(SOURCE_TABLE_CONF, START_DATE, END_DATE)

In [0]:
final_shipment_events_df = (shipment_events_df.filter(F.col("event_id").isNotNull())
        .withColumn('event_timestamp', F.to_timestamp(F.col("event_timestamp")))
            .withColumn('latitude', F.col("latitude").cast(DoubleType()))
            .withColumn('longitude', F.col("longitude").cast(DoubleType()))
            .withColumn('event_date', F.to_date(F.col("event_timestamp")))
            .withColumn('temperature_celsius', F.col("temperature_celsius").cast(DoubleType()))
            .select("event_id",
                    "event_timestamp",
                    "event_type",
                    "latitude",
                    "longitude",
                    "order_id",
                    "shipment_status",
                    "source_system",
                    "temperature_celsius",
                    "vehicle_id",
                    "event_date"))

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"
    primary_key = TARGET_TABLE_CONF.get('primary_key')

    (final_shipment_events_df
     .writeTo(target_table)
     .using("delta")
     .create()) #throws an error if run accidently after table creation
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY AUTO")
    spark.sql(f"ALTER TABLE {target_table} ALTER COLUMN {primary_key} SET NOT NULL")
    spark.sql(f"ALTER TABLE {target_table} ADD CONSTRAINT {primary_key}_pk PRIMARY KEY ({primary_key})")
    
    dbutils.notebook.exit(f"Initial Table: {target_table} Setup Finished Successfully")

In [0]:
upsert_to_delta(final_shipment_events_df, TARGET_TABLE_CONF)